# RAG Face-Off: Grokipedia vs. Wikipedia

Two independent Retrieval-Augmented Generation pipelines over two different knowledge bases,
answering the same question side by side.

```
                    ┌──────────────────────┐
   topic list ─────►│  Ingest              │
                    │  Grokipedia API      │──► raw docs (cached JSON)
                    │  Wikipedia MediaWiki │
                    └──────────┬───────────┘
                               ▼
                    ┌──────────────────────┐
                    │  Chunk (token-aware) │
                    └──────────┬───────────┘
                               ▼
                    ┌──────────────────────┐
                    │  Embed               │  text-embedding-3-small
                    └──────────┬───────────┘
                               ▼
             ┌─────────────────┴─────────────────┐
             ▼                                   ▼
   ┌───────────────────┐               ┌───────────────────┐
   │ Chroma collection │               │ Chroma collection │
   │   kb_grokipedia   │               │   kb_wikipedia    │
   └─────────┬─────────┘               └─────────┬─────────┘
             │        user question              │
             ▼                                   ▼
        retrieve top-k                      retrieve top-k
             │                                   │
             ▼                                   ▼
        gpt-4o-mini                         gpt-4o-mini
             │                                   │
             └──────────► side-by-side ◄─────────┘
```

**The whole point:** the *generator is identical* on both sides. Same model, same prompt,
same temperature, same `top_k`. The **only** variable is the corpus. So any difference you
see in the two answers is attributable to the knowledge base, not the LLM.

**Run order:** top to bottom. Steps 1–6 build the indexes (a few minutes and a few cents the
first time, then cached). Step 8 is the interactive comparison UI.

---
## Step 0 — Install dependencies

Run once per environment. Restart the kernel afterwards if `ipywidgets` was newly installed.

In [2]:
%pip install -q "openai>=1.40" "chromadb>=0.5" "grokipedia-api>=0.3.1" tiktoken ipywidgets requests pandas numpy python-dotenv

Note: you may need to restart the kernel to use updated packages.


---
## Step 1 — Configuration: pick your provider

**This notebook runs completely free.** Set `PROVIDER` below and everything downstream adapts.

| Provider | Cost | Needs | Notes |
|---|---|---|---|
| `groq` **(default)** | Free | API key, no credit card | Open models on very fast hardware. ~30 req/min, 1,000/day. |
| `ollama` | Free | Ollama installed locally | No key, no limits, no network. Slower on CPU. |
| `openai` | Paid | `OPENAI_API_KEY` | Best quality, costs a few cents. |

All three speak the **OpenAI-compatible chat API**, so switching providers is a `base_url`
change — the retrieval and generation code never changes.

### Embeddings are a separate decision

Groq and Ollama serve *chat* models, not embedding models. So the free path embeds **locally**
with MiniLM, which ChromaDB already bundles — no extra install, no key, and after a one-time
~80 MB model download it works offline forever.

### Two details that matter

**MiniLM truncates at 256 tokens.** Anything past that in a chunk is silently discarded — the
tail of every chunk would simply never be searchable. So the local path drops `CHUNK_TOKENS` to
220 automatically. This is the single most common way a free RAG build quietly breaks.

**Embedding models are not interchangeable.** MiniLM produces 384-dim vectors, OpenAI 1536-dim.
Mixing them in one collection is meaningless, so the collection name carries an `EMBED_TAG`
suffix. Switch backends and you get fresh collections rather than silent garbage.

In [1]:
import os
import re
import json
import time
import html
import hashlib
import textwrap
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Any, Optional, Tuple

# ==========================================================================
# CHOOSE YOUR PROVIDER: "groq" (free) | "ollama" (free, local) | "openai"
# ==========================================================================
PROVIDER = "groq"

PROVIDERS = {
    # Free tier, no credit card. Get a key at https://console.groq.com/keys
    "groq": {
        "base_url":    "https://api.groq.com/openai/v1",
        "key_env":     "GROQ_API_KEY",
        "chat_model":  "llama-3.3-70b-versatile",   # or "openai/gpt-oss-20b"
        "has_embeddings": False,
        "pause_s":     2.0,     # free tier is rate limited; be gentle in batch mode
    },
    # Fully local. Install from https://ollama.com then: ollama pull llama3.1:8b
    "ollama": {
        "base_url":    "http://localhost:11434/v1",
        "key_env":     None,
        "chat_model":  "llama3.1:8b",
        "has_embeddings": False,
        "pause_s":     0.0,
    },
    # Paid.
    "openai": {
        "base_url":    None,
        "key_env":     "OPENAI_API_KEY",
        "chat_model":  "gpt-4o-mini",
        "has_embeddings": True,
        "pause_s":     0.0,
    },
}

if PROVIDER not in PROVIDERS:
    raise ValueError(f"PROVIDER must be one of {list(PROVIDERS)}")

CFG          = PROVIDERS[PROVIDER]
CHAT_MODEL   = CFG["chat_model"]
TEMPERATURE  = 0.1
RATE_PAUSE_S = CFG["pause_s"]

# ---- Embedding backend ---------------------------------------------------
# "local"  -> MiniLM via ChromaDB (free, offline, 384 dims, 256-token limit)
# "openai" -> text-embedding-3-small (paid, 1536 dims)
EMBEDDING_BACKEND = "openai" if CFG["has_embeddings"] else "local"

if EMBEDDING_BACKEND == "local":
    EMBED_MODEL   = "all-MiniLM-L6-v2"
    EMBED_TAG     = "minilm"
    CHUNK_TOKENS  = 220     # MUST stay under MiniLM's 256-token input limit
    CHUNK_OVERLAP = 40
else:
    EMBED_MODEL   = "text-embedding-3-small"
    EMBED_TAG     = "oa3small"
    CHUNK_TOKENS  = 400
    CHUNK_OVERLAP = 60

# ---- Credentials ---------------------------------------------------------
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def _require_key(env_name: str) -> None:
    if env_name and not os.environ.get(env_name):
        from getpass import getpass
        os.environ[env_name] = getpass(f"{env_name}: ").strip()

_require_key(CFG["key_env"])
if EMBEDDING_BACKEND == "openai":
    _require_key("OPENAI_API_KEY")

# ---- Paths ---------------------------------------------------------------
DATA_DIR   = Path("data")
RAW_DIR    = DATA_DIR / "raw"
CHROMA_DIR = DATA_DIR / "chroma"
for d in (DATA_DIR, RAW_DIR, CHROMA_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- Retrieval knobs -----------------------------------------------------
TOP_K = 5                      # chunks fed to the LLM per side
MAX_CHARS_PER_DOC = 120_000    # guard against one enormous article

# ---- The corpus ----------------------------------------------------------
SOURCES = ["grokipedia", "wikipedia"]

TOPICS = [
    "Elon Musk",
    "Climate change",
    "COVID-19 pandemic origins",
    "Artificial intelligence",
    "Cryptocurrency",
    "Wikipedia",
]

print(f"Provider   : {PROVIDER}  ({CHAT_MODEL})")
print(f"Embeddings : {EMBEDDING_BACKEND}  ({EMBED_MODEL})")
print(f"Chunking   : {CHUNK_TOKENS} tokens / {CHUNK_OVERLAP} overlap")
print(f"Topics     : {len(TOPICS)}")
if PROVIDER != "openai":
    print("\nRunning free. No charges will be incurred.")

Provider   : groq  (llama-3.3-70b-versatile)
Embeddings : local  (all-MiniLM-L6-v2)
Chunking   : 220 tokens / 40 overlap
Topics     : 6

Running free. No charges will be incurred.


---
## Step 2 — Ingest the two knowledge bases

### Wikipedia
The MediaWiki Action API is public, stable, and needs no key. We do a search to resolve the
best-matching title (so `"COVID-19 pandemic origins"` lands on a real page), then pull the
plain-text extract of the whole article.

### Grokipedia
There is no official public API. We use the community `grokipedia-api` client, which wraps
Grokipedia's own internal `/api/full-text-search` and `/api/page-preview` endpoints. It is
unofficial and can break if xAI changes those endpoints — so every fetch is wrapped in
`try/except` and cached to disk. Once `data/raw/` is populated, the notebook never needs the
network again.

Both fetchers return the same `Document` shape, which is what lets the rest of the pipeline
stay source-agnostic.

In [2]:
import requests

USER_AGENT = "rag-grokipedia-vs-wikipedia/0.1 (educational comparison project)"

@dataclass
class Document:
    doc_id: str          # stable id, e.g. "wikipedia::Elon_Musk"
    source: str          # "wikipedia" | "grokipedia"
    topic: str           # the query we asked for
    title: str           # the title the source actually returned
    url: str
    text: str
    citations: List[Dict[str, str]] = field(default_factory=list)

    @property
    def n_chars(self) -> int:
        return len(self.text)


def _cache_path(source: str, topic: str) -> Path:
    key = hashlib.sha1(f"{source}::{topic}".encode()).hexdigest()[:12]
    slug = re.sub(r"[^a-z0-9]+", "-", topic.lower()).strip("-")[:40]
    return RAW_DIR / f"{source}__{slug}__{key}.json"


def _save(doc: Document) -> None:
    _cache_path(doc.source, doc.topic).write_text(
        json.dumps(asdict(doc), ensure_ascii=False, indent=2), encoding="utf-8"
    )


def _load(source: str, topic: str) -> Optional[Document]:
    p = _cache_path(source, topic)
    if p.exists():
        return Document(**json.loads(p.read_text(encoding="utf-8")))
    return None


def clean_text(text: str) -> str:
    text = text.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Grokipedia inlines citation markers like [12] or [12][13]; keep prose readable.
    text = re.sub(r"[ \t]*\[\d{1,3}\](?=\W|$)", "", text)
    return text.strip()

In [3]:
WIKI_API = "https://en.wikipedia.org/w/api.php"

def fetch_wikipedia(topic: str, use_cache: bool = True) -> Optional[Document]:
    if use_cache:
        cached = _load("wikipedia", topic)
        if cached:
            return cached

    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT})

    # 1. Resolve the best-matching article title.
    r = s.get(WIKI_API, params={
        "action": "query", "list": "search", "srsearch": topic,
        "srlimit": 1, "format": "json", "formatversion": 2,
    }, timeout=30)
    r.raise_for_status()
    hits = r.json().get("query", {}).get("search", [])
    if not hits:
        print(f"  [wikipedia] no match for {topic!r}")
        return None
    title = hits[0]["title"]

    # 2. Pull the full plain-text extract.
    r = s.get(WIKI_API, params={
        "action": "query", "prop": "extracts|info", "explaintext": 1,
        "redirects": 1, "titles": title, "inprop": "url",
        "format": "json", "formatversion": 2,
    }, timeout=60)
    r.raise_for_status()
    pages = r.json().get("query", {}).get("pages", [])
    if not pages or "extract" not in pages[0]:
        print(f"  [wikipedia] no extract for {title!r}")
        return None
    page = pages[0]

    doc = Document(
        doc_id=f"wikipedia::{page['title'].replace(' ', '_')}",
        source="wikipedia",
        topic=topic,
        title=page["title"],
        url=page.get("fullurl", f"https://en.wikipedia.org/wiki/{page['title'].replace(' ', '_')}"),
        text=clean_text(page["extract"])[:MAX_CHARS_PER_DOC],
        citations=[],
    )
    _save(doc)
    return doc

In [4]:
def fetch_grokipedia(topic: str, use_cache: bool = True) -> Optional[Document]:
    if use_cache:
        cached = _load("grokipedia", topic)
        if cached:
            return cached

    try:
        from grokipedia_api import GrokipediaClient
        from grokipedia_api.exceptions import GrokipediaError
    except ImportError:
        print("  [grokipedia] package not installed - run Step 0")
        return None

    try:
        with GrokipediaClient(timeout=45) as client:
            results = client.search(topic, limit=1).get("results", [])
            if not results:
                print(f"  [grokipedia] no match for {topic!r}")
                return None
            hit  = results[0]
            slug = hit.get("slug") or hit.get("title", "").replace(" ", "_")

            payload = client.get_page(slug, include_content=True)
            page    = payload.get("page", {}) or {}
            content = page.get("content") or ""
            if not content.strip():
                print(f"  [grokipedia] empty content for {slug!r}")
                return None

            citations = [
                {"title": str(c.get("title", ""))[:200], "url": str(c.get("url", ""))}
                for c in (page.get("citations") or [])
            ][:100]

            doc = Document(
                doc_id=f"grokipedia::{slug}",
                source="grokipedia",
                topic=topic,
                title=page.get("title") or hit.get("title") or topic,
                url=f"https://grokipedia.com/page/{slug}",
                text=clean_text(content)[:MAX_CHARS_PER_DOC],
                citations=citations,
            )
            _save(doc)
            return doc
    except GrokipediaError as e:
        print(f"  [grokipedia] API error for {topic!r}: {e}")
        return None
    except Exception as e:
        print(f"  [grokipedia] unexpected error for {topic!r}: {type(e).__name__}: {e}")
        return None

In [5]:
FETCHERS = {"wikipedia": fetch_wikipedia, "grokipedia": fetch_grokipedia}

def build_corpus(topics: List[str], use_cache: bool = True) -> Dict[str, List[Document]]:
    corpus = {src: [] for src in SOURCES}
    for topic in topics:
        print(f"Fetching: {topic}")
        for src in SOURCES:
            doc = FETCHERS[src](topic, use_cache=use_cache)
            if doc:
                corpus[src].append(doc)
                print(f"  [{src}] {doc.title!r} - {doc.n_chars:,} chars")
            time.sleep(0.4)          # be polite to both services
    return corpus

corpus = build_corpus(TOPICS)

print()
for src in SOURCES:
    total = sum(d.n_chars for d in corpus[src])
    print(f"{src:<12} {len(corpus[src])} docs, {total:,} chars")

Fetching: Elon Musk
  [grokipedia] 'Elon Musk' - 120,000 chars
  [wikipedia] 'Elon Musk' - 53,492 chars
Fetching: Climate change
  [grokipedia] 'Climate Change' - 113,249 chars
  [wikipedia] 'Climate change' - 64,283 chars
Fetching: COVID-19 pandemic origins
  [grokipedia] 'COVID-19 pandemic' - 120,000 chars
  [wikipedia] 'COVID-19 pandemic deaths' - 3,468 chars
Fetching: Artificial intelligence
  [grokipedia] 'Artificial Intelligence' - 66,546 chars
  [wikipedia] 'Artificial intelligence' - 87,106 chars
Fetching: Cryptocurrency
  [grokipedia] 'Cryptocurrency' - 106,868 chars
  [wikipedia] 'Cryptocurrency' - 72,111 chars
Fetching: Wikipedia
  [grokipedia] 'Wikipedia' - 120,000 chars


HTTPError: 429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Wikipedia&srlimit=1&format=json&formatversion=2

> **If Grokipedia returned nothing** (site down, endpoint changed, geo-blocked): drop article
> text into `data/raw/` manually using the same JSON shape as the Wikipedia files, and re-run
> the cell above. The cache is read first, so hand-written files are picked up transparently
> and the rest of the notebook works unchanged.

---
## Step 3 — Chunk the documents

Whole articles are far too big for a context window and far too coarse for retrieval — a
5,000-word article embedded as one vector averages away everything specific. We split into
chunks of `CHUNK_TOKENS` with `CHUNK_OVERLAP` tokens of overlap, both set by your provider
choice in Step 1 (220/40 for local MiniLM, 400/60 for OpenAI).

Token counting uses `tiktoken`'s `cl100k_base`. That is OpenAI's tokenizer, so for Llama or
MiniLM it is an *approximation* — close enough for sizing chunks, and it errs slightly
conservative, which is the safe direction when you have a hard input limit to respect.

The splitter is **paragraph-aware**: it packs whole paragraphs together until the token budget
is hit, so it almost never cuts a sentence in half. Only a single paragraph that is itself
larger than the budget gets a hard token split.

Overlap matters because a fact can straddle a boundary — the last sentence of chunk *n* and
the first of chunk *n+1*. Carrying 60 tokens forward means such facts stay retrievable.

In [ ]:
import tiktoken

ENC = tiktoken.get_encoding("cl100k_base")

def n_tokens(text: str) -> int:
    return len(ENC.encode(text))


def chunk_text(text: str,
               max_tokens: int = CHUNK_TOKENS,
               overlap: int = CHUNK_OVERLAP) -> List[str]:
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks: List[str] = []
    buf: List[str] = []
    buf_tokens = 0

    def flush():
        nonlocal buf, buf_tokens
        if buf:
            chunks.append("\n\n".join(buf))
            buf, buf_tokens = [], 0

    for para in paragraphs:
        t = n_tokens(para)

        # A single oversized paragraph: hard-split it on token boundaries.
        if t > max_tokens:
            flush()
            ids = ENC.encode(para)
            step = max(1, max_tokens - overlap)
            for i in range(0, len(ids), step):
                piece = ENC.decode(ids[i:i + max_tokens]).strip()
                if piece:
                    chunks.append(piece)
            continue

        if buf_tokens + t > max_tokens and buf:
            prev = "\n\n".join(buf)
            chunks.append(prev)
            # Carry the tail of the previous chunk forward as overlap.
            tail_ids = ENC.encode(prev)[-overlap:] if overlap > 0 else []
            tail = ENC.decode(tail_ids).strip()
            buf = [tail] if tail else []
            buf_tokens = len(tail_ids)

        buf.append(para)
        buf_tokens += t

    flush()
    return [c for c in chunks if c.strip()]

In [ ]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    source: str
    topic: str
    title: str
    url: str
    position: int
    text: str


def chunk_corpus(corpus: Dict[str, List[Document]]) -> Dict[str, List[Chunk]]:
    out = {src: [] for src in corpus}
    for src, docs in corpus.items():
        for doc in docs:
            for i, piece in enumerate(chunk_text(doc.text)):
                out[src].append(Chunk(
                    chunk_id=f"{doc.doc_id}::{i:04d}",
                    doc_id=doc.doc_id,
                    source=doc.source,
                    topic=doc.topic,
                    title=doc.title,
                    url=doc.url,
                    position=i,
                    text=piece,
                ))
    return out


chunks = chunk_corpus(corpus)

for src in SOURCES:
    cs = chunks[src]
    if not cs:
        print(f"{src:<12} 0 chunks")
        continue
    lens = [n_tokens(c.text) for c in cs]
    print(f"{src:<12} {len(cs):>4} chunks | "
          f"tokens min {min(lens)} / mean {sum(lens)//len(lens)} / max {max(lens)}")

# Eyeball one chunk to confirm the split looks sane.
if chunks[SOURCES[0]]:
    ex = chunks[SOURCES[0]][1]
    print(f"\n--- sample chunk: {ex.chunk_id} ---\n{ex.text[:600]}...")

---
## Step 4 — Embed

Two interchangeable backends behind one `embed_texts()` function.

**Local (free).** ChromaDB ships `DefaultEmbeddingFunction` — MiniLM-L6-v2 running through
ONNX Runtime on your CPU. 384 dimensions, no API key, no per-token cost, no rate limit. The
first call downloads ~80 MB; everything after that is offline. Indexing this corpus takes
roughly a minute on a modern laptop.

**OpenAI (paid).** `text-embedding-3-small`, 1536 dimensions. Meaningfully better on nuanced
semantic matching, but not required to make this project work.

The retry loop matters more than it looks: one transient failure partway through indexing
would otherwise leave you with a half-populated collection and silently degraded retrieval.

In [ ]:
from openai import OpenAI

def make_llm_client() -> OpenAI:
    # OpenAI-compatible client pointed at whichever provider is configured.
    kwargs: Dict[str, Any] = {}
    if CFG["base_url"]:
        kwargs["base_url"] = CFG["base_url"]
    # Ollama ignores the key but the SDK requires a non-empty string.
    kwargs["api_key"] = os.environ.get(CFG["key_env"] or "", "") or "not-needed"
    return OpenAI(**kwargs)

llm = make_llm_client()

# ---- Embedding backend ---------------------------------------------------
if EMBEDDING_BACKEND == "local":
    from chromadb.utils import embedding_functions
    _local_ef = embedding_functions.DefaultEmbeddingFunction()

    def _embed_batch(texts: List[str]) -> List[List[float]]:
        return [[float(x) for x in vec] for vec in _local_ef(texts)]

    EMBED_BATCH = 64
else:
    _embed_client = OpenAI()   # embeddings always go to OpenAI proper

    def _embed_batch(texts: List[str]) -> List[List[float]]:
        resp = _embed_client.embeddings.create(model=EMBED_MODEL, input=texts)
        return [d.embedding for d in resp.data]

    EMBED_BATCH = 96


def embed_texts(texts: List[str], batch_size: Optional[int] = None,
                verbose: bool = True) -> List[List[float]]:
    batch_size = batch_size or EMBED_BATCH
    vectors: List[List[float]] = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        for attempt in range(5):
            try:
                vectors.extend(_embed_batch(batch))
                break
            except Exception as e:
                if attempt == 4:
                    raise
                wait = 2 ** attempt
                print(f"  embed retry {attempt + 1}/4 after {type(e).__name__} - sleeping {wait}s")
                time.sleep(wait)
        if verbose:
            print(f"  embedded {min(start + batch_size, len(texts))}/{len(texts)}")
    return vectors


def embed_query(text: str) -> List[float]:
    return embed_texts([text], verbose=False)[0]


# Confirm the backend works and report the vector width.
_dim = len(embed_query("dimension probe"))
print(f"{EMBEDDING_BACKEND} embeddings OK - {_dim} dimensions")

---
## Step 5 — Build two Chroma collections

Two collections, `kb_grokipedia` and `kb_wikipedia`, in one persistent Chroma database on
disk. Keeping them separate — rather than one collection with a `source` metadata filter — is
deliberate: it guarantees zero cross-contamination between the two pipelines, and it means the
retrieval scores on each side are computed against that corpus alone.

We compute embeddings ourselves and hand them to Chroma rather than using its built-in
embedding function. That keeps the embedding step explicit and inspectable, and avoids version
drift in Chroma's OpenAI wrapper.

Distance is cosine (`hnsw:space`), so `similarity = 1 - distance`.

In [ ]:
import chromadb

chroma = chromadb.PersistentClient(path=str(CHROMA_DIR))

def collection_name(source: str) -> str:
    # EMBED_TAG keeps MiniLM (384d) and OpenAI (1536d) indexes strictly separate.
    return f"kb_{source}__{EMBED_TAG}"


def index_source(source: str, source_chunks: List[Chunk], reset: bool = False):
    name = collection_name(source)

    if reset:
        try:
            chroma.delete_collection(name)
        except Exception:
            pass

    col = chroma.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})

    existing = set()
    if col.count() > 0:
        existing = set(col.get(include=[])["ids"])

    todo = [c for c in source_chunks if c.chunk_id not in existing]
    if not todo:
        print(f"{name}: up to date ({col.count()} chunks)")
        return col

    print(f"{name}: embedding {len(todo)} new chunks...")
    vectors = embed_texts([c.text for c in todo])

    col.add(
        ids=[c.chunk_id for c in todo],
        documents=[c.text for c in todo],
        embeddings=vectors,
        metadatas=[{
            "doc_id":   c.doc_id,
            "source":   c.source,
            "topic":    c.topic,
            "title":    c.title,
            "url":      c.url,
            "position": c.position,
        } for c in todo],
    )
    print(f"{name}: now {col.count()} chunks")
    return col


collections = {src: index_source(src, chunks[src]) for src in SOURCES}

---
## Step 6 — Retrieval and generation

`retrieve()` embeds the question once and queries one collection. `answer()` formats the
retrieved chunks into a numbered context block and asks the model to answer *only* from that
block, citing chunk numbers.

The prompt does three things that keep the comparison honest:

1. **Grounding.** "Use only the context" — otherwise the model answers from its own parametric
   knowledge and you are no longer comparing knowledge bases.
2. **Forced citations.** `[1]`, `[2]` markers let you trace every claim back to a chunk, and
   make it obvious when the two sides are citing very different material.
3. **Explicit abstention.** If the context does not cover it, the model must say so. An honest
   "not covered" from one side is a real, informative result about that corpus.

In [ ]:
@dataclass
class Retrieved:
    rank: int
    chunk_id: str
    title: str
    url: str
    topic: str
    text: str
    similarity: float


def retrieve(question: str, source: str, top_k: int = TOP_K,
             query_vector: Optional[List[float]] = None) -> List[Retrieved]:
    col = collections[source]
    if col.count() == 0:
        return []
    qv = query_vector if query_vector is not None else embed_query(question)
    res = col.query(
        query_embeddings=[qv],
        n_results=min(top_k, col.count()),
        include=["documents", "metadatas", "distances"],
    )
    out = []
    for i, (doc, meta, dist) in enumerate(zip(
        res["documents"][0], res["metadatas"][0], res["distances"][0]
    )):
        out.append(Retrieved(
            rank=i + 1,
            chunk_id=res["ids"][0][i],
            title=meta.get("title", ""),
            url=meta.get("url", ""),
            topic=meta.get("topic", ""),
            text=doc,
            similarity=round(1.0 - float(dist), 4),
        ))
    return out

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful research assistant. Answer the user's question using ONLY the numbered "
    "context passages provided. Follow these rules strictly:\n"
    "1. Every factual claim must be supported by a passage, cited inline as [1], [2], etc.\n"
    "2. If the context does not contain enough information, say so explicitly instead of "
    "guessing or drawing on outside knowledge.\n"
    "3. If passages disagree with one another, surface the disagreement rather than smoothing "
    "it over.\n"
    "4. Match the framing and emphasis of the passages. Do not add caveats the sources do not "
    "make, and do not drop caveats the sources do make.\n"
    "5. Be concise: 3-6 sentences unless the question demands more."
)

USER_TEMPLATE = (
    "Context passages:\n"
    "{context}\n\n"
    "Question: {question}\n\n"
    "Answer using only the passages above, with inline [n] citations."
)


def format_context(hits: List[Retrieved], max_chars_each: int = 1800) -> str:
    parts = []
    for h in hits:
        body = h.text[:max_chars_each]
        parts.append(f"[{h.rank}] (from \"{h.title}\", similarity {h.similarity})\n{body}")
    return "\n\n".join(parts)


@dataclass
class RagAnswer:
    source: str
    question: str
    answer: str
    hits: List[Retrieved]
    latency_s: float
    prompt_tokens: int = 0
    completion_tokens: int = 0


def chat(messages: List[Dict[str, str]], max_retries: int = 5):
    # Free tiers rate limit aggressively - back off and retry rather than fail the run.
    for attempt in range(max_retries):
        try:
            return llm.chat.completions.create(
                model=CHAT_MODEL, temperature=TEMPERATURE, messages=messages,
            )
        except Exception as e:
            msg = str(e).lower()
            transient = any(s in msg for s in
                            ("rate limit", "429", "timeout", "overloaded", "503", "502"))
            if attempt == max_retries - 1 or not transient:
                raise
            wait = min(2 ** attempt * 3, 60)
            print(f"  {type(e).__name__} (likely rate limit) - retrying in {wait}s")
            time.sleep(wait)


def answer(question: str, source: str, top_k: int = TOP_K,
           query_vector: Optional[List[float]] = None) -> RagAnswer:
    t0 = time.time()
    hits = retrieve(question, source, top_k=top_k, query_vector=query_vector)

    if not hits:
        return RagAnswer(source, question,
                         "_No indexed content for this knowledge base._",
                         [], round(time.time() - t0, 2))

    resp = chat([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(
            context=format_context(hits), question=question)},
    ])
    usage = getattr(resp, "usage", None)
    return RagAnswer(
        source=source,
        question=question,
        answer=(resp.choices[0].message.content or "").strip(),
        hits=hits,
        latency_s=round(time.time() - t0, 2),
        prompt_tokens=getattr(usage, "prompt_tokens", 0) or 0,
        completion_tokens=getattr(usage, "completion_tokens", 0) or 0,
    )


# Quick smoke test - confirms the provider, key, and prompt all work end to end.
_probe = answer("What is this knowledge base mainly about?", "wikipedia", top_k=3)
print(f"[{PROVIDER} / {CHAT_MODEL}] {_probe.latency_s}s\n")
print(_probe.answer[:400])

---
## Step 7 — The comparison function

Embed the question **once** and reuse the vector for both retrievers — same query point in the
same embedding space, so neither side gets an accidental advantage from embedding jitter.

Beyond the two answers we compute three cheap diagnostics:

- **Answer similarity** — cosine similarity between embeddings of the two answers. High (>0.9)
  means the corpora agree. Low (<0.75) is where the interesting cases live.
- **Topic overlap** — did the two retrievers even pull from the same subject area?
- **Mean retrieval similarity** — how well each corpus actually covers the question. A low
  score means the answer is being generated from weak, tangential material.

In [ ]:
import numpy as np

def cosine(a, b) -> float:
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0


@dataclass
class Comparison:
    question: str
    results: Dict[str, RagAnswer]
    answer_similarity: float
    topic_overlap: List[str]

    def metrics_row(self) -> Dict[str, Any]:
        row: Dict[str, Any] = {"question": self.question,
                               "answer_similarity": self.answer_similarity}
        for src, r in self.results.items():
            sims = [h.similarity for h in r.hits]
            row[f"{src}_mean_retrieval_sim"] = round(sum(sims) / len(sims), 4) if sims else 0.0
            row[f"{src}_answer_words"] = len(r.answer.split())
            row[f"{src}_latency_s"] = r.latency_s
        return row


def compare(question: str, top_k: int = TOP_K) -> Comparison:
    qv = embed_query(question)                      # one embedding, both sides
    results = {}
    for i, src in enumerate(SOURCES):
        if i > 0 and RATE_PAUSE_S:
            time.sleep(RATE_PAUSE_S)                # stay inside free-tier limits
        results[src] = answer(question, src, top_k=top_k, query_vector=qv)

    texts = [r.answer for r in results.values()]
    if all(t.strip() for t in texts) and len(texts) == 2:
        va, vb = embed_texts(texts, verbose=False)
        sim = round(cosine(va, vb), 4)
    else:
        sim = 0.0

    topic_sets = [{h.topic for h in r.hits} for r in results.values()]
    overlap = sorted(set.intersection(*topic_sets)) if all(topic_sets) else []

    return Comparison(question, results, sim, overlap)

---
## Step 8 — Side-by-side comparison UI

The deliverable. Type a question, hit **Compare**, and get both answers next to each other with
their retrieved sources underneath.

> If the widgets render as plain text, run `jupyter nbextension enable --py widgetsnbextension`
> (classic notebook) or make sure `jupyterlab_widgets` is installed (JupyterLab), then restart
> the kernel.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

PALETTE = {
    "grokipedia": {"accent": "#1d9bf0", "label": "Grokipedia"},
    "wikipedia":  {"accent": "#7c3aed", "label": "Wikipedia"},
}


def _cite_html(hits: List[Retrieved]) -> str:
    rows = []
    for h in hits:
        snippet = html.escape(h.text[:220].replace("\n", " ")) + "..."
        rows.append(
            f"<details style='margin:4px 0;font-size:12px;'>"
            f"<summary style='cursor:pointer;'>"
            f"<b>[{h.rank}]</b> {html.escape(h.title)} "
            f"<span style='opacity:.6'>(sim {h.similarity})</span></summary>"
            f"<div style='margin:6px 0 6px 14px;opacity:.85;line-height:1.45'>{snippet}"
            f"<br><a href='{html.escape(h.url)}' target='_blank'>source &#8599;</a></div>"
            f"</details>"
        )
    return "".join(rows)


def _panel_html(res: RagAnswer) -> str:
    style = PALETTE.get(res.source, {"accent": "#666", "label": res.source})
    body = html.escape(res.answer).replace("\n", "<br>")
    sims = [h.similarity for h in res.hits]
    mean_sim = round(sum(sims) / len(sims), 3) if sims else 0.0
    return f'''
    <div style="flex:1;min-width:320px;border:1px solid #d8d8d8;border-top:4px solid {style['accent']};
                border-radius:8px;padding:14px 16px;background:#fff;">
      <div style="font-weight:700;color:{style['accent']};font-size:15px;margin-bottom:2px;">
        {style['label']}
      </div>
      <div style="font-size:11px;opacity:.6;margin-bottom:10px;">
        {len(res.hits)} chunks &middot; mean sim {mean_sim} &middot;
        {res.latency_s}s &middot; {res.prompt_tokens + res.completion_tokens} tokens
      </div>
      <div style="font-size:14px;line-height:1.6;color:#111;">{body}</div>
      <div style="margin-top:12px;border-top:1px solid #eee;padding-top:8px;">
        <div style="font-size:11px;text-transform:uppercase;letter-spacing:.5px;opacity:.55;
                    margin-bottom:4px;">Retrieved passages</div>
        {_cite_html(res.hits)}
      </div>
    </div>
    '''


def render_comparison(cmp_result: Comparison) -> HTML:
    sim = cmp_result.answer_similarity
    verdict, colour = (
        ("Strong agreement", "#16a34a") if sim >= 0.90 else
        ("Partial agreement", "#ca8a04") if sim >= 0.75 else
        ("Notable divergence", "#dc2626")
    )
    panels = "".join(_panel_html(cmp_result.results[s]) for s in SOURCES)
    shared = ", ".join(cmp_result.topic_overlap) or "none"
    header = f'''
    <div style="margin-bottom:10px;font-family:system-ui,-apple-system,sans-serif;">
      <div style="font-size:16px;font-weight:600;">{html.escape(cmp_result.question)}</div>
      <div style="font-size:12px;margin-top:4px;">
        <span style="color:{colour};font-weight:600;">{verdict}</span>
        <span style="opacity:.6;"> &middot; answer cosine similarity {sim}
        &middot; shared topics: {html.escape(shared)}</span>
      </div>
    </div>
    '''
    return HTML(
        "<div style=\"font-family:system-ui,-apple-system,sans-serif;\">"
        + header
        + f"<div style='display:flex;gap:14px;flex-wrap:wrap;align-items:stretch;'>{panels}</div>"
        + "</div>"
    )

In [ ]:
question_box = widgets.Textarea(
    value="What are the main criticisms of Elon Musk?",
    placeholder="Ask anything covered by the indexed topics...",
    layout=widgets.Layout(width="100%", height="72px"),
)
k_slider = widgets.IntSlider(value=TOP_K, min=1, max=10, description="top_k",
                             continuous_update=False)
run_button = widgets.Button(description="Compare", button_style="primary", icon="search")
out = widgets.Output()

EXAMPLES = [
    "What are the main criticisms of Elon Musk?",
    "What is the scientific consensus on the causes of climate change?",
    "What are the leading hypotheses for the origin of COVID-19?",
    "Is Wikipedia biased? What evidence is given?",
    "What are the main risks of cryptocurrency?",
]
example_dd = widgets.Dropdown(options=["-- example questions --"] + EXAMPLES,
                              layout=widgets.Layout(width="100%"))

def _on_example(change):
    if change["new"] and not change["new"].startswith("--"):
        question_box.value = change["new"]

example_dd.observe(_on_example, names="value")

def _on_click(_):
    q = question_box.value.strip()
    with out:
        clear_output(wait=True)
        if not q:
            print("Type a question first.")
            return
        print("Retrieving and generating...")
        try:
            result = compare(q, top_k=k_slider.value)
        except Exception as e:
            clear_output(wait=True)
            print(f"Error: {type(e).__name__}: {e}")
            return
        clear_output(wait=True)
        display(render_comparison(result))

run_button.on_click(_on_click)

display(widgets.VBox([
    widgets.HTML("<h3 style='margin:0 0 6px 0;'>RAG Face-Off &mdash; Grokipedia vs Wikipedia</h3>"),
    example_dd,
    question_box,
    widgets.HBox([run_button, k_slider]),
    out,
]))

---
## Step 9 — Batch evaluation

The UI is for exploration; this is for evidence. Run a fixed question set through both
pipelines and get a table you can sort, so you can point at *which* questions diverge instead
of relying on impressions.

Sort ascending by `answer_similarity`: the rows at the top are where the two encyclopedias
tell materially different stories, and those are the ones worth reading by hand.

In [ ]:
import pandas as pd

EVAL_QUESTIONS = [
    "What are the main criticisms of Elon Musk?",
    "What is the scientific consensus on the causes of climate change?",
    "What are the leading hypotheses for the origin of COVID-19?",
    "Is Wikipedia biased? What evidence is given?",
    "What are the main risks of cryptocurrency?",
    "What are the biggest risks posed by artificial intelligence?",
]

comparisons: List[Comparison] = []
for i, q in enumerate(EVAL_QUESTIONS):
    print(f"- {q}")
    if i > 0 and RATE_PAUSE_S:
        time.sleep(RATE_PAUSE_S)
    try:
        comparisons.append(compare(q))
    except Exception as e:
        print(f"  failed: {type(e).__name__}: {e}")

df = pd.DataFrame([c.metrics_row() for c in comparisons])
df = df.sort_values("answer_similarity").reset_index(drop=True)
df

In [ ]:
# Read the most divergent case in full.
if comparisons:
    most_divergent = min(comparisons, key=lambda c: c.answer_similarity)
    display(render_comparison(most_divergent))

In [ ]:
# Persist the run so results are reproducible and diffable in git.
export = [{
    "question": c.question,
    "answer_similarity": c.answer_similarity,
    "topic_overlap": c.topic_overlap,
    "answers": {src: {
        "answer": r.answer,
        "latency_s": r.latency_s,
        "hits": [{"rank": h.rank, "title": h.title, "url": h.url,
                  "similarity": h.similarity, "chunk_id": h.chunk_id} for h in r.hits],
    } for src, r in c.results.items()},
} for c in comparisons]

out_path = DATA_DIR / "comparison_results.json"
out_path.write_text(json.dumps(export, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote {out_path} ({len(export)} comparisons)")

---
## Where to take it next

**Make the comparison fairer.** Right now article length differs between sources, so one
corpus may simply have more chunks in play. Normalising chunk counts per topic removes that
confound.

**Add a third arm.** A no-RAG baseline — same model, same question, no context — shows how
much either knowledge base is actually contributing versus the model's own priors. It is often
a humbling result.

**Better retrieval.** Hybrid search (BM25 + dense, reciprocal-rank fusion) usually beats pure
dense retrieval on entity-heavy questions. A cross-encoder reranker over the top 20 is the
single highest-leverage upgrade after that.

**Automate the judging.** Have a stronger model score each answer for groundedness (is every
claim in the context?) and stance, then aggregate. Cheap to build, and it turns the divergence
number into something interpretable.

**Swap providers and re-run.** Because `EMBED_TAG` namespaces the collections and the prompt is
identical everywhere, you can set `PROVIDER = "openai"`, re-run Steps 1 and 4–9, and compare
free-stack output against paid-stack output on the same corpus. That tells you how much the
model tier actually matters for *your* questions, which is usually less than people assume for
grounded extractive answering.

### Honest caveats

- **Free-tier limits change without notice.** Groq and Google have both cut quotas abruptly.
  The numbers in Step 1 were accurate when written; check the provider console for live figures.
- **MiniLM is weaker than `text-embedding-3-small`**, mostly on paraphrase and nuance. If a
  question keeps retrieving tangential chunks, that is the first thing to suspect — raise
  `top_k` before blaming the corpus.

- The Grokipedia client is **unofficial**. It scrapes internal endpoints and may break without
  notice. Treat the on-disk cache as the real source of truth for any result you publish.
- Divergence between two answers is **not** evidence that either is wrong. It shows the corpora
  differ in coverage, framing, or emphasis — establishing which one is *accurate* requires
  checking the underlying citations yourself.
- Retrieval is the dominant source of variance here. Before concluding "corpus A is biased,"
  check whether corpus A simply had no good chunk to retrieve.
- Six topics is a demo, not a study. Any real claim needs a much broader topic set and multiple
  runs.